In [1]:
!pip install gtfparse
!pip install polars=='0.16.17'

In [2]:
!pip install pyarrow

In [3]:
!pip install anndata==0.8.0

In [5]:
from samalg import SAM
from Bio import SeqIO
from gtfparse import read_gtf
import pandas as pd
import pandas
import pyarrow
import pickle
import scanpy as sc
import numpy as np

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
#open scdata to see how many gene matches you have
dat = sc.read_h5ad('../../Testing_Raw_Dat_RNASEQ_Joined/tot_dat_DR_ncbi_joined_07152026.h5ad')

In [16]:
dat.var_names

Index(['42sp43', 'ATP6', 'ATP8', 'COX1', 'COX2', 'COX3', 'CYTB',
       'LOC100000155', 'LOC100000155_1', 'LOC100000275',
       ...
       'zte38', 'zufsp', 'zw10', 'zwi', 'zwilch', 'zyg11', 'zyg11l', 'zyx',
       'zzef1', 'zzz3'],
      dtype='object', length=25503)

In [17]:
input_file = open("../../cDNA_fasta/DR_ncbi_cdna_03142025.fa")
gene_dict = {}
for item in SeqIO.parse(input_file, "fasta"):
    print(item)
    break

ID: NM_173235.3
Name: NM_173235.3
Description: NM_173235.3 gene_id:rpl24 gene_name: transcript_name: chr:NC_007112.7 start:6642 end:11878 strand:-
Number of features: 0
Seq('AGGGTTCATTCCTATGTCAAATATATGTTTACTTCAAAAAAATATTTTACTTTA...TCA', SingleLetterAlphabet())


In [18]:
#pull longest gene
input_file = open("../../cDNA_fasta/DR_ncbi_cdna_03142025.fa")
gene_dict = {}
for item in SeqIO.parse(input_file, "fasta"):
    #manually change gene_ID to match which field you would like to see in your BLAST table
    gene_ID = item.description.split('gene_id:')[1].split(' ')[0]
    if gene_ID in gene_dict.keys():
        if len(item.seq) > len(gene_dict[gene_ID].seq):
            gene_dict[gene_ID] = item
    else:
        gene_dict[gene_ID] = item
len(gene_dict)

54144

In [19]:
len(set(dat.var_names) & set(gene_dict.keys()))

25502

In [20]:
for item in gene_dict.keys():
    gene_dict[item].id = item
    gene_dict[item].name = item

with open("../../DR_ncbi_genome_curated_08022026.fa", "w") as handle:
    SeqIO.write(gene_dict.values(), handle, "fasta") 

In [17]:
input_file = open("../../BLASTMAPPING/DR_ncbi_cdna_curated_03142025.fa")
fin_list = []
for item in SeqIO.parse(input_file, "fasta"):
    fin_list.append(item.name)

In [18]:
len(set(dat.var_names) & set(fin_list))

25502